# Compare Schedulability Test Results

This notebook compares two EDF-VD schedulability test result tables to identify entries where the `is_safe` verdict differs between runs. This helps detect any inconsistencies or improvements in the scheduling algorithm.

**Input Files:**
- `results-mcs_schedulability-2026-03-28_20-57-48.csv` (Original results)
- `results-mcs_schedulability-2026-04-15_20-29-49.csv` (New results)

**Goal:** Find taskset entries that produced different safety verdicts between the two test runs.

## 1. Load and Parse CSV Files

Import necessary libraries and load both result CSV files into DataFrames. We'll examine the structure and basic statistics of each dataset.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up paths
results_dir = Path('results')
file1 = results_dir / 'results-mcs_schedulability-2026-04-28_20-04-24.csv'
file2 = results_dir / 'results-mcs_schedulability-2026-05-01_02-06-07.csv'

# Load CSVs
print(f"Loading {file1}...")
df_original = pd.read_csv(file1, sep=';')
df_original = df_original.where(df_original['use_case'] == 'EDF-VD (exact)')
print(f"Loaded: {df_original.shape[0]} rows, {df_original.shape[1]} columns\n")

print(f"Loading {file2}...")
df_new = pd.read_csv(file2, sep=';')
print(f"Loaded: {df_new.shape[0]} rows, {df_new.shape[1]} columns\n")

# Display column names and first few rows
print("=" * 80)
print("ORIGINAL DATA (2026-04-28):")
print("=" * 80)
print(f"Columns: {list(df_original.columns)}\n")
print(df_original.head())
print(f"\nis_safe value counts:\n{df_original['is_safe'].value_counts()}")

print("\n" + "=" * 80)
print("NEW DATA (2026-05-01):")
print("=" * 80)
print(f"Columns: {list(df_new.columns)}\n")
print(df_new.head())
print(f"\nis_safe value counts:\n{df_new['is_safe'].value_counts()}")

Loading results/results-mcs_schedulability-2026-04-28_20-04-24.csv...
Loaded: 22000 rows, 19 columns

Loading results/results-mcs_schedulability-2026-05-01_02-06-07.csv...
Loaded: 11000 rows, 20 columns

ORIGINAL DATA (2026-03-28):
Columns: ['use_idlesim', 'safe_oracles', 'unsafe_oracles', 'use_case', 'scheduler', 'taskset_file', 'taskset_position', 'U', 'Uv', 'nbt', 'EDFVD_test', 'probability_of_HI', 'min_period', 'max_period', 'timeout', 'is_safe', 'automaton_depth', 'visited_count', 'duration_ns']

  use_idlesim safe_oracles      unsafe_oracles        use_case scheduler  \
0        True           []  ['hi-over-demand']  EDF-VD (exact)     edfvd   
1        True           []  ['hi-over-demand']  EDF-VD (exact)     edfvd   
2        True           []  ['hi-over-demand']  EDF-VD (exact)     edfvd   
3        True           []  ['hi-over-demand']  EDF-VD (exact)     edfvd   
4        True           []  ['hi-over-demand']  EDF-VD (exact)     edfvd   

                                  ta

## 2. Merge DataFrames on Common Keys

Merge the two DataFrames using taskset_file and taskset_position as the primary identifiers, along with other constant parameters (scheduler, use_case, etc.) to ensure we're comparing the same test runs.

In [4]:
# Define common identifier columns
key_columns = ['taskset_file', 'taskset_position', 'scheduler']

# Prepare dataframes for merging by adding suffixes
df_original_renamed = df_original.copy()
df_new_renamed = df_new.copy()

# Merge on key columns
merged = pd.merge(
    df_original_renamed,
    df_new_renamed,
    on=key_columns,
    how='inner',
    suffixes=('_original', '_new')
)

print(f"Merged successfully!")
print(f"Total merged rows: {len(merged)}")
print(f"Columns in merged data: {len(merged.columns)}")

# Show sample of merged data
print(f"\nFirst few rows of merged data:")
print(merged[[key_columns[0], key_columns[1], 'is_safe_original', 'is_safe_new']].head(10))

Merged successfully!
Total merged rows: 11000
Columns in merged data: 36

First few rows of merged data:
                                  taskset_file  taskset_position  \
0  outputs/20251022_055741-scheduling-rtss.txt               2.0   
1  outputs/20251022_055741-scheduling-rtss.txt               6.0   
2  outputs/20251022_055741-scheduling-rtss.txt               3.0   
3  outputs/20251022_055741-scheduling-rtss.txt              11.0   
4  outputs/20251022_055741-scheduling-rtss.txt              13.0   
5  outputs/20251022_055741-scheduling-rtss.txt               9.0   
6  outputs/20251022_055741-scheduling-rtss.txt              12.0   
7  outputs/20251022_055741-scheduling-rtss.txt              20.0   
8  outputs/20251022_055741-scheduling-rtss.txt              19.0   
9  outputs/20251022_055741-scheduling-rtss.txt               4.0   

  is_safe_original is_safe_new  
0             True        True  
1             True        True  
2             True        True  
3             

## 3. Identify Differing Results

Filter rows where the `is_safe` values differ between the original and new test runs. Create a summary showing which entries have conflicting outcomes.

In [9]:
# Find rows where is_safe values differ
differences = merged[merged['is_safe_original'] != merged['is_safe_new']].copy()

print("=" * 80)
print("DIFFERENCES IN is_safe VALUES")
print("=" * 80)
print(f"\nTotal entries with different is_safe verdicts: {len(differences)}")
print(f"Percentage of total merged entries: {100 * len(differences) / len(merged):.2f}%\n")

# Create a crosstab to show the transitions
if len(differences) > 0:
    transition_table = pd.crosstab(
        differences['is_safe_original'],
        differences['is_safe_new'],
        margins=True,
        margins_name='Total'
    )
    
    # Rename columns and index based on what we actually have
    col_mapping = {False: 'New: False', True: 'New: True', 'Total': 'Total'}
    row_mapping = {False: 'Original: False', True: 'Original: True', 'Total': 'Total'}
    
    # Rename only the columns/index that exist
    transition_table = transition_table.rename(
        columns={col: col_mapping.get(col, str(col)) for col in transition_table.columns},
        index={idx: row_mapping.get(idx, str(idx)) for idx in transition_table.index}
    )
    
    print("Transition Table (Original -> New):")
    print(transition_table)
    print()
    
    # Show sample of differences
    print("Sample of entries with differing is_safe values:")
    print("-" * 80)
    sample_cols = ['taskset_file', 'taskset_position', 'scheduler', 'U', 'Uv', 
                   'is_safe_original', 'is_safe_new', 'automaton_depth_original', 
                   'automaton_depth_new', 'visited_count_original', 'visited_count_new']
    available_cols = [col for col in sample_cols if col in differences.columns]
    display(differences[available_cols])
else:
    print("✓ No differences found - all is_safe verdicts are consistent!")

DIFFERENCES IN is_safe VALUES

Total entries with different is_safe verdicts: 243
Percentage of total merged entries: 2.21%

Transition Table (Original -> New):
Empty DataFrame
Columns: []
Index: []

Sample of entries with differing is_safe values:
--------------------------------------------------------------------------------


,taskset_file,taskset_position,scheduler,is_safe_original,is_safe_new,automaton_depth_original,automaton_depth_new,visited_count_original,visited_count_new
324,outputs/20251022_055741-scheduling-rtss.txt,78.0,edfvd,True,NaN,28.0,NaN,421916.0,NaN
393,outputs/20251022_055741-scheduling-rtss.txt,391.0,edfvd,True,NaN,27.0,NaN,31255.0,NaN
755,outputs/20251022_055741-scheduling-rtss.txt,708.0,edfvd,True,NaN,25.0,NaN,106956.0,NaN
1161,outputs/20251022_055741-scheduling-rtss.txt,1132.0,edfvd,True,NaN,28.0,NaN,89750.0,NaN
1203,outputs/20251022_055741-scheduling-rtss.txt,1217.0,edfvd,True,NaN,19.0,NaN,5698.0,NaN
...,...,...,...,...,...,...,...,...,...
10268,outputs/20251022_055741-scheduling-rtss.txt,9876.0,edfvd,True,NaN,115.0,NaN,4924054.0,NaN
10545,outputs/20251022_055741-scheduling-rtss.txt,9918.0,edfvd,True,NaN,141.0,NaN,6748538.0,NaN
10604,outputs/20251022_055741-scheduling-rtss.txt,9803.0,edfvd,False,NaN,182.0,NaN,8625280.0,NaN
10824,outputs/20251022_055741-scheduling-rtss.txt,9935.0,edfvd,True,NaN,144.0,NaN,8451274.0,NaN


## 4. Analyze and Visualize Differences

Examine the differing entries for patterns. Compare automaton_depth and visited_count for cases with different is_safe results, and create visualizations to show the nature and frequency of discrepancies.

In [6]:
if len(differences) > 0:
    print("=" * 80)
    print("DETAILED ANALYSIS OF DIFFERENCES")
    print("=" * 80)
    
    # Analyze automaton depth and visited count changes
    differences['depth_diff'] = differences['automaton_depth_new'] - differences['automaton_depth_original']
    differences['visited_diff'] = differences['visited_count_new'] - differences['visited_count_original']
    
    print("\nAutomaton Depth Changes (New - Original):")
    print(differences['depth_diff'].describe())
    
    print("\nVisited Count Changes (New - Original):")
    print(differences['visited_diff'].describe())
    
    # Analyze by verdict change direction
    print("\n" + "-" * 80)
    print("Analysis by Change Type:")
    print("-" * 80)
    
    changed_false_to_true = differences[
        (differences['is_safe_original'] == False) & 
        (differences['is_safe_new'] == True)
    ]
    print(f"\nBecame SAFE (False -> True): {len(changed_false_to_true)} entries")
    if len(changed_false_to_true) > 0:
        print(f"  Avg depth change: {changed_false_to_true['depth_diff'].mean():.2f}")
        print(f"  Avg visited count change: {changed_false_to_true['visited_diff'].mean():.2f}")
    
    changed_true_to_false = differences[
        (differences['is_safe_original'] == True) & 
        (differences['is_safe_new'] == False)
    ]
    print(f"\nBecame UNSAFE (True -> False): {len(changed_true_to_false)} entries")
    if len(changed_true_to_false) > 0:
        print(f"  Avg depth change: {changed_true_to_false['depth_diff'].mean():.2f}")
        print(f"  Avg visited count change: {changed_true_to_false['visited_diff'].mean():.2f}")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Automaton Depth Changes
    axes[0, 0].hist(differences['depth_diff'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Automaton Depth Change (New - Original)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Automaton Depth Changes')
    axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No change')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Visited Count Changes
    axes[0, 1].hist(differences['visited_diff'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Visited Count Change (New - Original)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Visited Count Changes')
    axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No change')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Comparison by U (Utilization)
    if 'U' in differences.columns:
        colors = ['red' if x else 'green' for x in differences['is_safe_new']]
        axes[1, 0].scatter(differences['U'], differences['automaton_depth_new'], 
                          c=colors, alpha=0.6, s=50)
        axes[1, 0].set_xlabel('Utilization (U)')
        axes[1, 0].set_ylabel('Automaton Depth (New)')
        axes[1, 0].set_title('Automaton Depth vs Utilization\n(Green=Safe, Red=Unsafe)')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Verdict change summary
    change_types = ['False→True\n(Became Safe)', 'True→False\n(Became Unsafe)']
    change_counts = [len(changed_false_to_true), len(changed_true_to_false)]
    bars = axes[1, 1].bar(change_types, change_counts, color=['green', 'red'], alpha=0.7, edgecolor='black')
    axes[1, 1].set_ylabel('Number of Entries')
    axes[1, 1].set_title('Summary of Verdict Changes')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar, count in zip(bars, change_counts):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{int(count)}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('comparison_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualization saved to comparison_analysis.png")

else:
    print("\nNo differences to analyze - all verdicts match!")

DETAILED ANALYSIS OF DIFFERENCES

Automaton Depth Changes (New - Original):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: depth_diff, dtype: float64

Visited Count Changes (New - Original):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: visited_diff, dtype: float64

--------------------------------------------------------------------------------
Analysis by Change Type:
--------------------------------------------------------------------------------

Became SAFE (False -> True): 0 entries

Became UNSAFE (True -> False): 0 entries


/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/matplotlib/axes/_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/matplotlib/axes/_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


ValueError: autodetected range of [nan, nan] is not finite

## 5. Export Comparison Results

Save a new CSV file containing only the rows with differing `is_safe` values, including both the original and new results side-by-side for easy review and analysis.

In [7]:
if len(differences) > 0:
    # Select key columns for export
    export_columns = [
        'taskset_file', 'taskset_position', 'scheduler', 'use_case',
        'U', 'Uv', 'nbt',
        'is_safe_original', 'is_safe_new',
        'automaton_depth_original', 'automaton_depth_new',
        'visited_count_original', 'visited_count_new',
        'duration_ns_original', 'duration_ns_new'
    ]
    
    # Filter to only available columns
    available_export_cols = [col for col in export_columns if col in differences.columns]
    
    # Create export dataframe
    export_df = differences[available_export_cols].copy()
    
    # Add a column indicating the change type
    export_df['change_type'] = export_df.apply(
        lambda row: 'False→True (Became Safe)' if row['is_safe_original'] == False and row['is_safe_new'] == True 
        else 'True→False (Became Unsafe)' if row['is_safe_original'] == True and row['is_safe_new'] == False 
        else 'Unknown',
        axis=1
    )
    
    # Sort by taskset file and position
    export_df = export_df.sort_values(['taskset_file', 'taskset_position'])
    
    # Save to CSV
    output_file = 'results/differing_is_safe_results.csv'
    export_df.to_csv(output_file, index=False)
    
    print("=" * 80)
    print("EXPORT RESULTS")
    print("=" * 80)
    print(f"\n✓ Exported {len(export_df)} entries with differing is_safe values")
    print(f"✓ Saved to: {output_file}")
    print(f"\nExport includes:")
    print(f"  - Test identifiers (taskset file, position, scheduler)")
    print(f"  - Taskset properties (U, Uv, nbt)")
    print(f"  - Safety verdicts (original and new)")
    print(f"  - Automaton metrics (depth and visited counts)")
    print(f"  - Execution times (original and new)")
    print(f"  - Change type classification")
    
    print(f"\nFirst 10 rows of exported data:")
    print(export_df[['taskset_file', 'taskset_position', 'is_safe_original', 
                      'is_safe_new', 'change_type']].head(10).to_string())
    
else:
    print("\n✓ No entries to export - all is_safe verdicts are consistent!")
    print("No differences file created.")

EXPORT RESULTS

✓ Exported 243 entries with differing is_safe values
✓ Saved to: results/differing_is_safe_results.csv

Export includes:
  - Test identifiers (taskset file, position, scheduler)
  - Taskset properties (U, Uv, nbt)
  - Safety verdicts (original and new)
  - Automaton metrics (depth and visited counts)
  - Execution times (original and new)
  - Change type classification

First 10 rows of exported data:
                                     taskset_file  taskset_position is_safe_original is_safe_new change_type
324   outputs/20251022_055741-scheduling-rtss.txt              78.0             True         NaN     Unknown
393   outputs/20251022_055741-scheduling-rtss.txt             391.0             True         NaN     Unknown
755   outputs/20251022_055741-scheduling-rtss.txt             708.0             True         NaN     Unknown
1161  outputs/20251022_055741-scheduling-rtss.txt            1132.0             True         NaN     Unknown
1335  outputs/20251022_055741-sche